In [ ]:
from pathlib import Path

import pandas as pd
import spacy
from tqdm.auto import tqdm


# ============================================================================
# 02. CONTENT-WORD POS MASKING
# ============================================================================

BASE_DIR = Path("..").resolve()
INPUT_CSV_PATH = BASE_DIR / "data/raw/combined_nli_raw.csv"
OUTPUT_CSV_PATH = BASE_DIR / "data/processed/combined_nli_masked.csv"

MASK_MAP = {
    "NOUN": "N",
    "PROPN": "P",
    "VERB": "V",
    "ADJ": "A",
    "ADV": "R",
}

REQUIRED_COLUMNS = {"doc_id", "text", "label", "dataset", "split"}


def mask_document(doc) -> str:
    masked_tokens = []
    for token in doc:
        if token.text in {"<P>", "<S>"}:
            masked_tokens.append(token.text)
        elif token.pos_ in MASK_MAP:
            masked_tokens.append(MASK_MAP[token.pos_])
        else:
            masked_tokens.append(token.text)
    return " ".join(masked_tokens)


print("Loading combined corpus...")
df = pd.read_csv(INPUT_CSV_PATH)
missing_columns = REQUIRED_COLUMNS - set(df.columns)
if missing_columns:
    raise ValueError(f"Input is missing required columns: {sorted(missing_columns)}")

df["text"] = df["text"].fillna("").astype(str)
df["label"] = df["label"].astype(str).str.strip().str.upper()
df["dataset"] = df["dataset"].astype(str).str.strip().str.upper()
df["split"] = df["split"].astype(str).str.strip().str.lower()

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])

print(f"Masking {len(df)} essays...")
masked_texts = []
token_counts = []

for doc in tqdm(nlp.pipe(df["text"], batch_size=256), total=len(df)):
    masked_texts.append(mask_document(doc))
    token_counts.append(len(doc))

df["masked_text"] = masked_texts
df["token_count"] = token_counts
df["masked_token_count"] = df["masked_text"].str.split().str.len()

empty_masked = df["masked_text"].str.len().eq(0).sum()
if empty_masked:
    raise ValueError(f"{empty_masked} rows produced empty masked text.")

OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Saved masked corpus: {OUTPUT_CSV_PATH}")
print(df[["doc_id", "label", "dataset", "split", "text", "masked_text"]].head(2))
